### Sales Development Representative

- We'll visit Sendgrid (sendgrid.com) for API Key and Single sender verification.

In [1]:
# Let's import the nessary libraries and set up the environment variables for our project.
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [2]:
# The usual start point
load_dotenv(override=True)

True

### Step1: Agent workflow

In [3]:
instructions1 = "You are a sales agent working for PenthouseAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for PenthouseAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for PenthouseAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [4]:
# Make agents with names, instructions and model
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [5]:
# Run the agents with a prompt and stream the results
result = Runner.run_streamed(sales_agent1, input="Write a cold email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC 2 Compliance Process with PenthouseAI

Dear [Recipient's Name],

I hope this message finds you well. 

As organizations increasingly prioritize data security and compliance, the challenge of maintaining SOC 2 standards can be daunting. At PenthouseAI, we understand the complexities involved in ensuring compliance and preparing for audits. 

Our AI-powered platform simplifies the SOC 2 compliance process, allowing you to focus on your core business while we manage the intricacies of documentation, monitoring, and audit readiness. With PenthouseAI, you can:

- Automate compliance tracking and reporting
- Reduce the time and resources required for audits
- Stay ahead of regulatory changes effortlessly

I would love to schedule a brief call to discuss how PenthouseAI can support your organization in achieving and maintaining SOC 2 compliance efficiently.

Thank you for your time, and I look forward to the opportunity to connect.

Best regards,

[Your Name]  
[Y

In [6]:
# Now let's run all three agents in parallel and compare their outputs.
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, input=message),
        Runner.run(sales_agent2, input=message),
        Runner.run(sales_agent3, input=message)
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Simplify Your SOC 2 Compliance Journey with PenthouseAI

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent PenthouseAI, where we specialize in streamlining SOC 2 compliance through our advanced AI-powered SaaS tool.

In today’s regulatory landscape, ensuring compliance can be a complex and time-consuming feat. Our solution not only simplifies the preparation process but also helps organizations reduce the risk of non-compliance and improve overall audit readiness.

Key benefits of our platform include:

- **Automated Evidence Collection:** Effortlessly gather and manage necessary documentation.
- **Real-time Compliance Monitoring:** Stay ahead of potential issues with continuous oversight.
- **Customizable Frameworks:** Tailor compliance workflows to align with your specific needs.

Companies like [Example Client 1] and [Example Client 2] have significantly enhanced their compliance processes with our tool, reducing their audit

In [7]:
# Finally, let's create an agent that picks the best email from the three outputs.
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from this given options. \
    Imagine you are a customer and pick the one you are most likely to respond to. \
    Do not give an explanation; reply with the selected email only."
)

In [8]:
# Run the picker agent with the three outputs and get the best email.
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, input=message),
        Runner.run(sales_agent2, input=message),
        Runner.run(sales_agent3, input=message)
    )

outputs = [result.final_output for result in results]

emails = "Cold sales emails:\n\n".join(outputs)

best_email_result = await Runner.run(sales_picker, input=emails)

print(f"Best sales email:\n{best_email_result.final_output}")

Best sales email:
Subject: Simplify Your SOC 2 Compliance with AI

Hi [Recipient's Name],

I hope this message finds you well! I'm [Your Name] from PenthouseAI, and we specialize in streamlining SOC 2 compliance with our AI-powered SaaS tool.

Our platform automates documentation and audit preparation, saving your team time and reducing compliance stress. Companies like [Example Company] have seen significant improvements in their audit processes.

Can we schedule a quick 15-minute call to discuss how we can help you achieve seamless compliance?

Best,  
[Your Name]  
[Your Job Title]  
[Your Phone Number]  
[PenthouseAI Website]


### Steps 2 and 3: Tools and Agent interactions

In [9]:
# Now we have the best email, let's send it out using SendGrid. We will create a function tool for this. 
@function_tool
def send_email(body: str):
    """Send out an email with the given body to all sales prospects"""
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
    from_email = Email("iwanttotestanapp@gmail.com")
    to_email = To("ctrlplusstyle@gmail.com")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales Email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [10]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x779180476840>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [11]:
# We can also convert an Agent into a tool
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x779180191370>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

### So now we can gather all the tools together:
A tool for each 3 email-writing agents

And a tool for our function to send emails

In [12]:
# Define the description for the tools
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x779180191ca0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Fa

### Now it's time for our Sales Manager - our planning agent

In [13]:
instructions = """
You are a Sales Manager at PenthouseAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=instructions,
    model="gpt-4o-mini",
    tools=tools
)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales Manager Agent"):
    result = await Runner.run(sales_manager, input=message)

### Handoffs

In [14]:
# The sales manager agent will generate three drafts using the sales agent tools
# select the best one, and then send it using the send_email tool. 
# We can check the output and the email sent to verify that it worked as expected.
subject_instructions = "You can write a subject for a cold sales email. \
    You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
    You are given a text email body which might have some mardown \
        and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email Subject Writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML Email Body Converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [16]:
# Now we can create a new sales manager agent that uses these tools to write a subject
# convert the email body to HTML, and then send the email.
@function_tool
def send_html_email(subject: str, html_body: str):
    """Send out an HTML email with the given subject and body to all sales prospects"""
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
    from_email = Email("iwanttotestanapp@gmail.com")
    to_email = To("ctrlplusstyle@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [17]:
# Define the tools that the sales manager agent will use
tools = [subject_tool, html_tool, send_html_email]

In [19]:
# Define the instructions for the sales manager agent that uses the new tools 
instructions = "You are an email formatter and Writer. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the hmtl_converter tool to convert the body to html. \
Finally, you use the send_html_email tool to send the email with the subject and html body."

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it out"
)

### Now we have 3 tools and 1 handoff 

In [21]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [22]:
# Now we can create a new sales manager agent that uses the sales agent tools to generate drafts
# selects the best one, and then hands it off to the email manager agent to format and send.
sales_manager_instructions = """
You are a Sales Manager at PenthouseAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini"
)

message = "Send out a cold sales email addresed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, input=message)


Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
